In [ ]:
# [1] 설치 + 설정 + 데이터 불러오기
!pip install -q yfinance
import random
import yfinance as yf
import pandas as pd
import numpy as np

OLD22 = [
 "005930.KS","000660.KS","035420.KS","005380.KS","051910.KS","006400.KS",
 "035720.KS","068270.KS","105560.KS","012330.KS","009150.KS","034020.KS",
 "003670.KS","096770.KS","247540.KQ","086520.KQ","196170.KQ","041510.KQ",
 "263750.KQ","058470.KQ","293490.KQ","357780.KQ",
]
NEW = [
 "000270.KS","055550.KS","066570.KS","017670.KS","015760.KS","032830.KS",
 "028260.KS","011200.KS","010130.KS","090430.KS","086790.KS","003550.KS",
 "034730.KS","018260.KS","010950.KS","036570.KS","251270.KS","011170.KS",
 "009540.KS","329180.KS","042660.KS","012450.KS","079550.KS","267260.KS",
 "064350.KS","028300.KQ","214150.KQ","145020.KQ","039030.KQ","240810.KQ",
 "036930.KQ","078340.KQ","035900.KQ","122870.KQ","112040.KQ","066970.KQ",
 "277810.KQ","403870.KQ","095340.KQ","005290.KQ","108320.KQ",
]
START = "2018-01-01"
SPLIT = pd.Timestamp("2023-01-01")
FEE = 0.0015        # 편도 수수료+슬리피지 가정
STOP_BELOW = 0.04   # 돌파선 아래 손절 폭
ATR_MULT = 3.0      # 절반 익절 후 트레일링: 고점 - ATR x 3
EXC = ["247540.KQ"] # 에코프로비엠 제외 (포함하려면 [])


def dl(t):
    d = yf.download(t, start=START, auto_adjust=True, progress=False)
    if d is None or d.empty:
        return None
    if isinstance(d.columns, pd.MultiIndex):
        d.columns = d.columns.droplevel(1)
    d.index = pd.to_datetime(d.index).tz_localize(None)
    return d


DATA = {}
for t in OLD22 + NEW:
    try:
        d = dl(t)
        if d is not None:
            d = d[["Open", "High", "Low", "Close", "Volume"]].dropna()
            d = d[d.Volume > 0]
            if len(d) > 400:
                DATA[t] = d
                continue
    except Exception:
        pass
    print("건너뜀:", t)

m = dl("^KS11")
MKT_OK = (m.Close > m.Close.rolling(60).mean()).astype(bool)
print(f"{len(DATA)}종목 로드 완료")

In [ ]:
# [2] 핵심 함수: 매매 신호/체결 + 계좌 시뮬레이션
def get_trades(d, above200=False, rs=False, vol=0, mkt=False, chase=None):
    O, H, L, C = (d[c].values for c in ["Open", "High", "Low", "Close"])
    Cs, Hs, Ls, Vs = d.Close, d.High, d.Low, d.Volume
    n = len(d)
    resist = Hs.rolling(60).max().shift(1)
    broke = (Cs > resist) & (Cs.shift(1) <= resist)
    if vol:
        broke &= Vs > Vs.rolling(20).mean().shift(1) * vol
    if mkt:
        broke &= MKT_OK.reindex(d.index).ffill().fillna(False).astype(bool)
    if above200:
        broke &= Cs > Cs.rolling(200).mean()
    if rs:
        idx_r = (m.Close / m.Close.shift(120) - 1).reindex(d.index).ffill()
        broke &= (Cs / Cs.shift(120) - 1) > idx_r
    lvl = resist.shift(1)
    conf = broke.shift(1, fill_value=False) & (Ls >= lvl * 0.99) & (Cs > lvl)
    nxt = Hs.rolling(250).max().shift(1).values
    pc = Cs.shift(1)
    tr = pd.concat([Hs - Ls, (Hs - pc).abs(), (Ls - pc).abs()], axis=1).max(axis=1)
    atr = tr.rolling(14).mean().values
    lv = lvl.values
    out, free = [], 0
    for i in np.where(conf.values)[0]:
        j = i + 1
        if j >= n or j < free:
            continue
        entry, stop = O[j], lv[i] * (1 - STOP_BELOW)
        if chase and entry > lv[i] * (1 + chase):
            continue
        if not stop < entry:
            continue
        risk = (entry - stop) / entry
        if not 0.01 <= risk <= 0.12:
            continue
        tgt = nxt[i]
        if np.isnan(tgt) or tgt < entry * (1 + risk):
            tgt = entry * (1 + 2 * risk)
        half, hi, tstop, part = False, entry, 0.0, 0.0
        xk, xp = n - 1, C[-1]
        for k in range(j, n):
            cur = max(tstop, entry) if half else stop
            if k > j and O[k] <= cur:
                xk, xp = k, O[k]
                break
            if L[k] <= cur:
                xk, xp = k, cur
                break
            if (not half) and H[k] >= tgt:
                half, part = True, tgt / entry - 1
            hi = max(hi, H[k])
            if half:
                tstop = max(tstop, hi - ATR_MULT * atr[k])
        ret = (0.5 * part + 0.5 * (xp / entry - 1)) if half else (xp / entry - 1)
        ret -= 2 * FEE
        free = xk + 1
        out.append(dict(entry=d.index[j], exit=d.index[xk], ret=ret, risk=risk))
    return out


def simulate(trades, max_pos=10, risk_pct=0.01, max_alloc=0.25, init=1_000_000):
    cash, openp, curve, skipped = init, [], [], 0

    def settle(dt):
        nonlocal cash
        for p in sorted(openp, key=lambda p: p["exit"]):
            if p["exit"] <= dt:
                cash += p["alloc"] * (1 + p["ret"])
                openp.remove(p)
                curve.append((p["exit"], cash + sum(q["alloc"] for q in openp)))

    for t in trades:
        settle(t["entry"])
        eq = cash + sum(p["alloc"] for p in openp)
        if len(openp) >= max_pos:
            skipped += 1
            continue
        alloc = min(eq * risk_pct / t["risk"], eq * max_alloc, cash)
        if alloc < eq * 0.02:
            skipped += 1
            continue
        cash -= alloc
        openp.append(dict(t, alloc=alloc))
    settle(pd.Timestamp("2100-01-01"))
    s = pd.Series([v for _, v in curve],
                  index=pd.DatetimeIndex([x for x, _ in curve]))
    s = s.groupby(level=0).last()
    s = pd.concat([pd.Series([init], index=[trades[0]["entry"]]), s])
    yrs = (s.index[-1] - s.index[0]).days / 365.25
    return dict(final=s.iloc[-1], cagr=(s.iloc[-1] / init) ** (1 / yrs) - 1,
                mdd=(s / s.cummax() - 1).min(), n=len(trades) - skipped,
                skip=skipped, curve=s)


print("함수 준비 완료")

In [ ]:
# [3] 테마 정의 + 테마 강도 + 필터
THEMES = {
 "반도체": ["005930.KS","000660.KS","009150.KS","058470.KQ","357780.KQ",
          "039030.KQ","240810.KQ","036930.KQ","403870.KQ","095340.KQ",
          "005290.KQ","108320.KQ"],
 "2차전지": ["051910.KS","006400.KS","003670.KS","096770.KS",
           "086520.KQ","247540.KQ","066970.KQ"],
 "바이오": ["068270.KS","196170.KQ","028300.KQ","214150.KQ","145020.KQ"],
 "게임엔터": ["263750.KQ","293490.KQ","036570.KS","251270.KS","078340.KQ",
           "112040.KQ","041510.KQ","035900.KQ","122870.KQ"],
 "조선방산": ["009540.KS","329180.KS","042660.KS","012450.KS",
           "079550.KS","064350.KS"],
 "금융": ["105560.KS","055550.KS","032830.KS","086790.KS"],
 "자동차": ["005380.KS","012330.KS","000270.KS"],
 "전력에너지": ["034020.KS","267260.KS","015760.KS"],
}
TH_OF = {t: th for th, L in THEMES.items() for t in L}

CL = pd.DataFrame({t: d.Close for t, d in DATA.items()}).ffill()
R20 = CL / CL.shift(20) - 1
NH5 = (CL >= CL.rolling(60).max()).astype(float).rolling(5).max()
score, cnt = {}, {}
for th, L in THEMES.items():
    cols = [t for t in L if t in CL.columns]
    if len(cols) >= 3:
        score[th] = R20[cols].mean(axis=1)   # 테마 20일 평균 수익률
        cnt[th] = NH5[cols].sum(axis=1)      # 최근 5일 신고가 종목 수
SC, CN = pd.DataFrame(score), pd.DataFrame(cnt)
RANK = SC.rank(axis=1, ascending=False)


def cache_trades(vol):
    return {tk: get_trades(d, vol=vol) for tk, d in DATA.items() if tk in TH_OF}


def pick(cache, exclude=(), topk=3, cluster=2):
    out = []
    for tk, trs in cache.items():
        th = TH_OF[tk]
        if tk in exclude or th not in SC.columns:
            continue
        for x in trs:
            pos = RANK.index.searchsorted(x["entry"]) - 1   # 진입 전날 기준
            if pos < 0:
                continue
            if topk and not (RANK[th].iloc[pos] <= topk and SC[th].iloc[pos] > 0):
                continue
            if cluster and not CN[th].iloc[pos] >= cluster:
                continue
            out.append(x)
    return sorted(out, key=lambda x: x["entry"])


def go(tr):
    return simulate(tr) if len(tr) >= 30 else None


CACHE = {v: cache_trades(v) for v in [1.5, 2.0, 2.5]}
print("테마:", list(SC.columns))

In [ ]:
# [4] 메인 결과: 테마 상위3 + 클러스터2 + 거래량 1.5배
r = go(pick(CACHE[1.5], EXC, topk=3, cluster=2))
s = r["curve"]
yrs = (s.index[-1] - s.index[0]).days / 365.25
bm = m.Close[m.index >= s.index[0]]
print(f"기간: {s.index[0].date()} ~ {s.index[-1].date()} ({yrs:.1f}년)")
print(f"1,000,000원 -> {r['final']:,.0f}원 (연평균 {r['cagr']:.1%})")
print(f"최대 낙폭 {r['mdd']:.1%} | 매매 {r['n']}회 (한도로 {r['skip']}회 패스)")
print(f"같은 기간 코스피 보유: {bm.iloc[-1] / bm.iloc[0] - 1:+.1%}\n")
yr = s.groupby(s.index.year).last()
prev = yr.shift(1)
prev.iloc[0] = 1_000_000
print("연도별 수익률")
print((yr / prev - 1).map("{:+.1%}".format).to_string())

In [ ]:
# [5] 견고성 테스트: 파라미터 27조합 / 비용 2배 / 에코프로 포함 / 종목 절반 제거
print("거래량 / 테마순위 / 클러스터 -> 연평균, MDD, 매매")
cagrs = []
for v in [1.5, 2.0, 2.5]:
    for k in [2, 3, 4]:
        for c in [1, 2, 3]:
            rr = go(pick(CACHE[v], EXC, topk=k, cluster=c))
            if rr is None:
                print(f"{v} / {k} / {c}: 신호 부족")
                continue
            cagrs.append(rr["cagr"])
            print(f"{v} / {k} / {c}: {rr['cagr']:>6.1%}  {rr['mdd']:>6.1%}  {rr['n']}회")
sr = pd.Series(cagrs)
print(f"\n조합 {len(sr)}개 | 플러스 {int((sr > 0).sum())}개 | "
      f"중앙값 {sr.median():.1%} | 최저 {sr.min():.1%} | 최고 {sr.max():.1%}")

print("\n-- 비용 2배 --")
FEE = 0.003
c_cost = cache_trades(1.5)
FEE = 0.0015
for name, tr in [("비용 1배", pick(CACHE[1.5], EXC)), ("비용 2배", pick(c_cost, EXC))]:
    rr = go(tr)
    print(f"{name}: 연평균 {rr['cagr']:.1%} | MDD {rr['mdd']:.1%} | {rr['n']}회")

print("\n-- 에코프로비엠 포함 --")
rr = go(pick(CACHE[1.5], []))
print(f"포함: 연평균 {rr['cagr']:.1%} | MDD {rr['mdd']:.1%}")

print("\n-- 종목 절반 랜덤 제거 (8회) --")
pool = [t for t in CACHE[1.5] if t not in EXC]
res = []
for seed in range(8):
    random.seed(seed)
    drop = set(random.sample(pool, len(pool) // 2))
    rr = go(pick(CACHE[1.5], drop | set(EXC)))
    if rr is None:
        print(f"시도 {seed + 1}: 신호 부족")
        continue
    res.append(rr["cagr"])
    print(f"시도 {seed + 1}: 연평균 {rr['cagr']:>6.1%} | MDD {rr['mdd']:>6.1%} | {rr['n']}회")
if res:
    print(f"플러스 {sum(x > 0 for x in res)}/{len(res)} | 중앙값 {np.median(res):.1%}")

In [ ]:
# [6] 워크포워드: 매년 '그 해 이전 데이터'로만 최고 조합을 골라 다음 해 성적 확인
import itertools

GRID = list(itertools.product([1.5, 2.0, 2.5], [2, 3, 4], [1, 2, 3]))
ALL = {g: pick(CACHE[g[0]], EXC, topk=g[1], cluster=g[2]) for g in GRID}
FIXED = (2.0, 3, 2)


def year_ret(tr, a, b):
    sub = [x for x in tr if a <= x["entry"] < b]
    if len(sub) < 8:
        return None
    return simulate(sub)["final"] / 1_000_000 - 1


def fmt(v):
    return "  -  " if v is None else f"{v:+.1%}"


print("연도 | 과거최고로 고른 조합 | 그해 성적 | 고정조합 | 전체평균")
picked, fixed = [], []
for y in range(2021, 2027):
    a, b = pd.Timestamp(f"{y}-01-01"), pd.Timestamp(f"{y+1}-01-01")
    best, bs = None, -9
    for g, tr in ALL.items():
        past = [x for x in tr if x["exit"] < a]
        if len(past) < 40:
            continue
        sc = simulate(past)["cagr"]
        if sc > bs:
            best, bs = g, sc
    if best is None:
        continue
    p = year_ret(ALL[best], a, b)
    f = year_ret(ALL[FIXED], a, b)
    vals = [year_ret(t, a, b) for t in ALL.values()]
    vals = [v for v in vals if v is not None]
    avg = np.mean(vals) if vals else None
    if p is not None:
        picked.append(p)
    if f is not None:
        fixed.append(f)
    print(f"{y} | {best} | {fmt(p)} | {fmt(f)} | {fmt(avg)}")
if picked and fixed:
    print(f"\n워크포워드 평균 {np.mean(picked):+.1%} | 고정조합 평균 {np.mean(fixed):+.1%}")